In [2]:
!nvidia-smi

Fri Feb 27 09:06:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
|  0%   30C    P8             28W /  450W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
import bitsandbytes

/workspace/lora_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
classifier = pipeline(
    'zero-shot-classification',
    model = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device = 0
)
text = "Врач был внимательным, но не на столько чтобы прям вау, просто норм, советовать сложно, ну а так пойдет!"
candidate_labels = ["позитивный", "негативный", "нейтральный"]

result = classifier(text, candidate_labels, multi_label=False)

print(f"текст: {text}")
print(f"класс:  {result['labels'][0]} ({result['scores'][0]:.2%})")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 573.93it/s, Materializing param=pooler.dense.weight]                                       
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


текст: Врач был внимательным, но не на столько чтобы прям вау, просто норм, советовать сложно, ну а так пойдет!
класс:  нейтральный (44.98%)
